# Stage 1 — Knowledge Graph Construction

Builds a typed multi-relational knowledge graph per dataset (NYC / TKY) from the
Stage 0 outputs (`poi_metadata_*.csv`, `train/val/test_*.csv`).

**Nodes:** POI, Category (one node per hierarchy level), Locality, Region

**Edges:**
- `SUBCATEGORY_OF` — category chain, child → parent
- `HAS_CATEGORY` — POI → its leaf category node
- `LOCATED_IN` — POI → locality → region, chained the same way as category
- `IS_NEAR_TO` — k-NN geographic proximity, weighted `1/(1+distance_km)`
- `FOLLOWED_BY` — sequential check-in transitions from `train_df` only, weighted by transition count

**Output:** `kg_{DATASET}.graphml`, `kg_{DATASET}.gpickle`, plus an edge-count summary.


## 0. Setup

In [31]:
import os
import pickle
from itertools import combinations

import numpy as np
import pandas as pd
import networkx as nx
from sklearn.neighbors import NearestNeighbors


## 1. Config

`DATASET` toggles which dataset's CSVs get loaded, same pattern as the Stage 0 notebook.
Fill in the `"TKY"` paths once those CSVs are exported the same way.


In [32]:
DATASET    = "NYC"   # "NYC" or "TKY"
OUTPUT_DIR = "/kaggle/working"

K_NEIGHBORS = 10          # k for the IS_NEAR_TO proximity graph
EARTH_RADIUS_KM = 6371.0

PATHS = {
    "NYC": {
        "poi_meta": "/kaggle/input/datasets/yosrkharrat/data-nyc/poi_metadata_NYC.csv",
        "train":    "/kaggle/input/datasets/yosrkharrat/data-nyc/train_NYC.csv",
        "val":      "/kaggle/input/datasets/yosrkharrat/data-nyc/val_NYC.csv",
        "test":     "/kaggle/input/datasets/yosrkharrat/data-nyc/test_NYC.csv",
    },
    "TKY": {
        "poi_meta": "/kaggle/input/datasets/yosrkharrat/tky-data/poi_metadata_TKY.csv",   # TODO: fix path once exported
        "train":    "/kaggle/input/datasets/yosrkharrat/tky-data/train_TKY.csv",           # TODO
        "val":      "/kaggle/input/datasets/yosrkharrat/tky-data/val_TKY.csv",             # TODO
        "test":     "/kaggle/input/datasets/yosrkharrat/tky-data/test_TKY.csv",            # TODO
    },
}

paths = PATHS[DATASET]
print(f"Building KG for {DATASET}")
for k, v in paths.items():
    print(f"  {k:9s} -> {v}  (exists: {os.path.exists(v)})")


Building KG for NYC
  poi_meta  -> /kaggle/input/datasets/yosrkharrat/data-nyc/poi_metadata_NYC.csv  (exists: True)
  train     -> /kaggle/input/datasets/yosrkharrat/data-nyc/train_NYC.csv  (exists: True)
  val       -> /kaggle/input/datasets/yosrkharrat/data-nyc/val_NYC.csv  (exists: True)
  test      -> /kaggle/input/datasets/yosrkharrat/data-nyc/test_NYC.csv  (exists: True)


## 2. Load Stage 0 outputs

In [33]:
poi_meta = pd.read_csv(paths["poi_meta"])
train_df = pd.read_csv(paths["train"])
val_df   = pd.read_csv(paths["val"])
test_df  = pd.read_csv(paths["test"])

if "utc_time" in train_df.columns:
    train_df["utc_time"] = pd.to_datetime(train_df["utc_time"], utc=True, errors="coerce")

print(f"poi_meta : {poi_meta.shape}  cols={list(poi_meta.columns)}")
print(f"train_df : {train_df.shape}")
print(f"val_df   : {val_df.shape}")
print(f"test_df  : {test_df.shape}")


poi_meta : (5120, 9)  cols=['venue_id', 'name', 'category', 'locality', 'region', 'description', 'poi_idx', 'latitude', 'longitude']
train_df : (117736, 9)
val_df   : (14657, 9)
test_df  : (15146, 9)


In [34]:
# Sanity check inherited from Stage 0: poi_idx should be contiguous, no gaps
_no_gaps = set(poi_meta["poi_idx"].astype(int)) == set(range(len(poi_meta)))
print("poi_idx range has no gaps:", _no_gaps)

_missing_coords = poi_meta[["latitude", "longitude"]].isna().any(axis=1).sum()
print(f"POIs missing lat/lon: {_missing_coords} / {len(poi_meta)}")

_missing_loc = poi_meta["locality"].isna().sum()
_missing_reg = poi_meta["region"].isna().sum()
print(f"POIs missing locality: {_missing_loc} / {len(poi_meta)}  "
      f"(expected > 0 -- placeholder/unmatched rows have no locality/region)")
print(f"POIs missing region:   {_missing_reg} / {len(poi_meta)}")


poi_idx range has no gaps: True
POIs missing lat/lon: 0 / 5120
POIs missing locality: 705 / 5120  (expected > 0 -- placeholder/unmatched rows have no locality/region)
POIs missing region:   689 / 5120


## 3. Node ID helpers

Node IDs are namespaced by type so the same string (e.g. a category leaf name
that happens to match a locality name) can never collide across node types.
Category nodes are keyed on the **full path prefix**, not just the leaf label,
so that e.g. `"Nightlife > Bar"` and `"Retail > Bar"` (if that existed) would
be two distinct nodes rather than merged into one.


In [35]:
def poi_id(poi_idx) -> str:
    return f"poi:{int(poi_idx)}"

def cat_id(path_prefix: str) -> str:
    return f"cat:{path_prefix}"

def loc_id(locality: str) -> str:
    return f"loc:{locality.strip()}"

def reg_id(region: str) -> str:
    return f"reg:{region.strip()}"


## 4. Category hierarchy — `SUBCATEGORY_OF` + `HAS_CATEGORY`

`category` holds the full chain (e.g. `"Dining and Drinking > Bar"`), confirmed
in the Stage 0 notebook. Each level becomes a node; edges point child → parent,
mirroring the reference `add_category_hierarchy` behaviour.


In [36]:
def add_category_hierarchy(G: nx.MultiDiGraph, category_path):
    """Add the SUBCATEGORY_OF chain for one category path and return the leaf node id.
    Returns None if category_path is null/empty (placeholder rows with no FSQ match
    still carry venue_category_name as a single-level category, so this should be rare)."""
    if pd.isna(category_path) or not str(category_path).strip():
        return None

    levels = [lvl.strip() for lvl in str(category_path).split(">")]
    prev_node = None
    cumulative = []
    for lvl in levels:
        cumulative.append(lvl)
        path_prefix = " > ".join(cumulative)
        node = cat_id(path_prefix)
        if node not in G:
            G.add_node(node, type="Category", name=lvl, level=len(cumulative))
        if prev_node is not None:
            # key=relation makes repeat calls for the same (node, prev_node) pair
            # (e.g. two POIs sharing an ancestor category) overwrite the same edge
            # instead of stacking a new parallel edge in the MultiDiGraph.
            G.add_edge(node, prev_node, key="SUBCATEGORY_OF", relation="SUBCATEGORY_OF")
        prev_node = node
    return prev_node  # leaf node id


## 5. Location hierarchy — `LOCATED_IN`

Same chaining pattern as category: `POI -> locality -> region`. Placeholder
(unmatched) POIs have null locality/region (expected, per Stage 0) and simply
get no `LOCATED_IN` edge -- they still get `HAS_CATEGORY`, so no POI is left
totally disconnected.


In [37]:
def add_location_hierarchy(G: nx.MultiDiGraph, locality, region):
    """Add the region node (if present) and the locality -> region edge (if both
    present), returning the leaf node (locality if present, else region, else None)."""
    region_node = None
    if pd.notna(region) and str(region).strip():
        region_node = reg_id(region)
        if region_node not in G:
            G.add_node(region_node, type="Region", name=str(region).strip())

    locality_node = None
    if pd.notna(locality) and str(locality).strip():
        locality_node = loc_id(locality)
        if locality_node not in G:
            G.add_node(locality_node, type="Locality", name=str(locality).strip())
        if region_node is not None:
            G.add_edge(locality_node, region_node, key="LOCATED_IN", relation="LOCATED_IN")

    return locality_node if locality_node is not None else region_node


## 6. Proximity edges — `IS_NEAR_TO`

k-NN over POI coordinates using haversine distance (correct for lat/lon, unlike
raw Euclidean distance on degrees). Weight = `1/(1+distance_km)`. Treated as a
symmetric relation: both directions are added for every k-NN pair found, deduped
so a mutual nearest-neighbour pair doesn't get double-added.


In [38]:
def add_proximity_edges(G: nx.MultiDiGraph, poi_meta: pd.DataFrame, k: int = K_NEIGHBORS):
    coords_df = poi_meta.dropna(subset=["latitude", "longitude"])[["poi_idx", "latitude", "longitude"]]
    if len(coords_df) < 2:
        print("Not enough POIs with coordinates to build proximity edges.")
        return

    idx = coords_df["poi_idx"].astype(int).to_numpy()
    coords_rad = np.radians(coords_df[["latitude", "longitude"]].to_numpy())

    k_eff = min(k + 1, len(coords_df))  # +1 because a point is its own nearest neighbour
    nbrs = NearestNeighbors(n_neighbors=k_eff, metric="haversine").fit(coords_rad)
    distances, indices = nbrs.kneighbors(coords_rad)

    seen = set()
    n_edges = 0
    for row_i, (dist_row, idx_row) in enumerate(zip(distances, indices)):
        src_poi = idx[row_i]
        for dist, row_j in zip(dist_row, idx_row):
            if row_j == row_i:
                continue
            dst_poi = idx[row_j]
            dist_km = dist * EARTH_RADIUS_KM
            weight = 1.0 / (1.0 + dist_km)

            for a, b in ((src_poi, dst_poi), (dst_poi, src_poi)):
                key = (a, b)
                if key in seen:
                    continue
                seen.add(key)
                G.add_edge(poi_id(a), poi_id(b), key="IS_NEAR_TO", relation="IS_NEAR_TO",
                           weight=float(weight), distance_km=float(dist_km))
                n_edges += 1

    print(f"IS_NEAR_TO: added {n_edges} directed edges from k={k} (k_eff={k_eff}) neighbours "
          f"over {len(coords_df)} located POIs")


## 7. Sequential transitions — `FOLLOWED_BY`

Built from `train_df` **only** (never val/test, to avoid leaking future check-ins
into the graph the downstream model will condition on). Consecutive check-ins per
user, sorted chronologically, are aggregated into a transition-count-weighted edge.


In [39]:
def add_sequential_transition_edges(G: nx.MultiDiGraph, train_df: pd.DataFrame):
    if "utc_time" not in train_df.columns:
        raise ValueError("train_df has no utc_time column -- cannot order check-ins chronologically")

    sort_cols = ["user_id", "utc_time"]
    train_sorted = train_df.sort_values(sort_cols)

    transition_counts = {}
    for _, g in train_sorted.groupby("user_id"):
        pois = g["poi_idx"].astype(int).tolist()
        for a, b in zip(pois[:-1], pois[1:]):
            if a == b:
                continue  # skip self-transitions (repeat check-in at same POI back-to-back)
            key = (a, b)
            transition_counts[key] = transition_counts.get(key, 0) + 1

    for (a, b), cnt in transition_counts.items():
        G.add_edge(poi_id(a), poi_id(b), key="FOLLOWED_BY", relation="FOLLOWED_BY", weight=float(cnt))

    print(f"FOLLOWED_BY: added {len(transition_counts)} directed edges "
          f"from {train_sorted['user_id'].nunique()} users' trajectories")


## 8. Build the full graph

In [40]:
G = nx.MultiDiGraph()

# --- POI nodes --------------------------------------------------------
for _, row in poi_meta.iterrows():
    G.add_node(
        poi_id(row["poi_idx"]),
        type="POI",
        poi_idx=int(row["poi_idx"]),
        name=row.get("name", None),
        latitude=row.get("latitude", None),
        longitude=row.get("longitude", None),
    )

# --- Category hierarchy + HAS_CATEGORY --------------------------------
n_has_category = 0
for _, row in poi_meta.iterrows():
    leaf = add_category_hierarchy(G, row.get("category"))
    if leaf is not None:
        G.add_edge(poi_id(row["poi_idx"]), leaf, key="HAS_CATEGORY", relation="HAS_CATEGORY")
        n_has_category += 1
print(f"HAS_CATEGORY: added {n_has_category} edges "
      f"({len(poi_meta) - n_has_category} POIs have no category -- should be ~0)")

# --- Location hierarchy + LOCATED_IN -----------------------------------
n_located_in = 0
for _, row in poi_meta.iterrows():
    leaf = add_location_hierarchy(G, row.get("locality"), row.get("region"))
    if leaf is not None:
        G.add_edge(poi_id(row["poi_idx"]), leaf, key="LOCATED_IN", relation="LOCATED_IN")
        n_located_in += 1
print(f"LOCATED_IN (POI -> locality/region): added {n_located_in} edges "
      f"({len(poi_meta) - n_located_in} POIs have neither -- expected for placeholder rows)")

# --- Proximity ----------------------------------------------------------
add_proximity_edges(G, poi_meta, k=K_NEIGHBORS)

# --- Sequential transitions ---------------------------------------------
add_sequential_transition_edges(G, train_df)

print(f"\nGraph built: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges")


HAS_CATEGORY: added 5120 edges (0 POIs have no category -- should be ~0)
LOCATED_IN (POI -> locality/region): added 4431 edges (689 POIs have neither -- expected for placeholder rows)
IS_NEAR_TO: added 63144 directed edges from k=10 (k_eff=11) neighbours over 5120 located POIs
FOLLOWED_BY: added 59541 directed edges from 1073 users' trajectories

Graph built: 5,764 nodes, 132,799 edges


## 9. Save outputs

In [41]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

graphml_path  = os.path.join(OUTPUT_DIR, f"kg_{DATASET}.graphml")
gpickle_path  = os.path.join(OUTPUT_DIR, f"kg_{DATASET}.gpickle")

# graphml requires scalar (str/int/float/bool) attributes -- sanitize None -> "" first
G_export = G.copy()
for _, data in G_export.nodes(data=True):
    for k, v in list(data.items()):
        if v is None:
            data[k] = ""
for _, _, data in G_export.edges(data=True):
    for k, v in list(data.items()):
        if v is None:
            data[k] = ""

nx.write_graphml(G_export, graphml_path)
with open(gpickle_path, "wb") as f:
    pickle.dump(G, f)

print(f"Saved: {graphml_path}")
print(f"Saved: {gpickle_path}")


Saved: /kaggle/working/kg_NYC.graphml
Saved: /kaggle/working/kg_NYC.gpickle


## 10. Sanity checks / edge-count summary

In [42]:
from collections import Counter

node_type_counts = Counter(nx.get_node_attributes(G, "type").values())
print("--- Node counts by type ---")
for t, c in node_type_counts.items():
    print(f"  {t:10s}: {c:,}")

relation_counts = Counter(d["relation"] for _, _, d in G.edges(data=True))
print("\n--- Edge counts by relation ---")
for r, c in relation_counts.items():
    print(f"  {r:15s}: {c:,}")

print(f"\nTotal nodes: {G.number_of_nodes():,}")
print(f"Total edges: {G.number_of_edges():,}")


--- Node counts by type ---
  POI       : 5,120
  Category  : 500
  Region    : 2
  Locality  : 142

--- Edge counts by relation ---
  HAS_CATEGORY   : 5,120
  LOCATED_IN     : 4,577
  IS_NEAR_TO     : 63,144
  FOLLOWED_BY    : 59,541
  SUBCATEGORY_OF : 417

Total nodes: 5,764
Total edges: 132,799


In [43]:
# Duplicate-edge check: with key=relation on every add_edge call, the number of
# (u, v, relation) triples should equal the number of (u, v) pairs per relation --
# i.e. no relation type should have more edges than unique node pairs it connects.
dupe_found = False
for r in relation_counts:
    pairs = {(u, v) for u, v, d in G.edges(data=True) if d["relation"] == r}
    if len(pairs) != relation_counts[r]:
        dupe_found = True
        print(f"  [DUPLICATE EDGES] {r}: {relation_counts[r]:,} edges but only "
              f"{len(pairs):,} unique (u,v) pairs -- add_edge is missing key=relation somewhere")
if not dupe_found:
    print("No duplicate parallel edges detected -- edge count per relation matches unique (u,v) pair count.")


No duplicate parallel edges detected -- edge count per relation matches unique (u,v) pair count.


In [44]:
# Every POI should have at least a HAS_CATEGORY edge -- isolated POIs would break
# Stage 5's "fallback to centroid" assumption if there were more than a handful.
poi_nodes = [n for n, d in G.nodes(data=True) if d.get("type") == "POI"]
isolated_pois = [n for n in poi_nodes if G.degree(n) == 0]
print(f"POI nodes: {len(poi_nodes):,}")
print(f"Isolated POI nodes (no edges at all): {len(isolated_pois)} "
      f"(should be 0 -- every POI has a category)")
if isolated_pois:
    print("  e.g.:", isolated_pois[:10])


POI nodes: 5,120
Isolated POI nodes (no edges at all): 0 (should be 0 -- every POI has a category)


In [45]:
# Spot-check: pull one POI and print its immediate neighbourhood across relation types
sample_poi = poi_nodes[0]
print(f"Neighbourhood of {sample_poi} ({G.nodes[sample_poi].get('name')}):\n")
for u, v, d in G.out_edges(sample_poi, data=True):
    print(f"  --{d['relation']}--> {v}  (w={d.get('weight', 1.0):.3f})")
for u, v, d in G.in_edges(sample_poi, data=True):
    print(f"  <--{d['relation']}-- {u}  (w={d.get('weight', 1.0):.3f})")


Neighbourhood of poi:0 (Fat Cat):

  --HAS_CATEGORY--> cat:Arts and Entertainment > Performing Arts Venue > Music Venue > Jazz and Blues Venue  (w=1.000)
  --LOCATED_IN--> loc:New York  (w=1.000)
  --IS_NEAR_TO--> poi:198  (w=0.985)
  --IS_NEAR_TO--> poi:3802  (w=0.980)
  --IS_NEAR_TO--> poi:1535  (w=0.969)
  --IS_NEAR_TO--> poi:3564  (w=0.956)
  --IS_NEAR_TO--> poi:618  (w=0.955)
  --IS_NEAR_TO--> poi:3372  (w=0.943)
  --IS_NEAR_TO--> poi:371  (w=0.938)
  --IS_NEAR_TO--> poi:202  (w=0.932)
  --IS_NEAR_TO--> poi:207  (w=0.920)
  --IS_NEAR_TO--> poi:306  (w=0.915)
  --IS_NEAR_TO--> poi:902  (w=0.860)
  --IS_NEAR_TO--> poi:2885  (w=0.909)
  --FOLLOWED_BY--> poi:30  (w=1.000)
  --FOLLOWED_BY--> poi:274  (w=1.000)
  --FOLLOWED_BY--> poi:1840  (w=1.000)
  --FOLLOWED_BY--> poi:2299  (w=1.000)
  --FOLLOWED_BY--> poi:992  (w=1.000)
  --FOLLOWED_BY--> poi:4423  (w=1.000)
  --FOLLOWED_BY--> poi:631  (w=1.000)
  --FOLLOWED_BY--> poi:1600  (w=1.000)
  --FOLLOWED_BY--> poi:5097  (w=1.000)
  --FOLLO